# PEFT Basics: Prompt Tuning Qwen3.5-2B-Base


Практический notebook по Parameter-Efficient Fine-Tuning (PEFT) на `Qwen/Qwen3.5-2B-Base`.

Используется **Prompt Tuning**: веса базовой модели остаются замороженными, а обучаются только виртуальные prompt embeddings. Это позволяет сначала понять принцип PEFT, а затем перейти к LoRA и QLoRA.

Задача - бинарная классификация тональности SST-2, сформулированная как causal language modeling: модель должна генерировать `positive` или `negative`.


## Теоретическая часть


### Что такое fine-tuning


Предобученная языковая модель уже умеет работать с языком и хранит большое количество общих знаний, но это не означает, что она оптимально решает конкретную прикладную задачу.

**Fine-tuning** — это продолжение обучения pretrained model на специализированных данных. Он позволяет адаптировать модель к определённому домену, формату ответа, стилю, классификации, извлечению информации или другой downstream-задаче без обучения модели с нуля.


### Full fine-tuning


При классическом **full fine-tuning** практически все параметры модели остаются trainable (изменяются).

Градиенты проходят через всю сеть, optimizer обновляет все или почти все веса, а для каждой новой задачи обычно получается отдельная полностью дообученная копия модели.

Упрощённо:

```text
Pretrained model
      │
      ▼
Все параметры trainable
      │
      ▼
Training
      │
      ▼
Полная новая версия модели
```


### Почему full fine-tuning дорогой


Для моделей с миллиардами параметров стоимость обучения определяется не только размером файла с весами.

Во время обучения в GPU memory находятся:

- параметры модели;
- gradients;
- optimizer states;
- activations;
- временные tensors и buffers.

Например, 2 миллиарда параметров в BF16 занимают примерно 4 GB только для хранения самих весов. Реальный training footprint значительно больше.

Кроме того, если одну модель адаптировать к десяти задачам через full fine-tuning, приходится хранить десять почти полных копий модели.


### Что такое PEFT


**PEFT — Parameter-Efficient Fine-Tuning** — семейство методов, которые позволяют адаптировать большие pretrained models, обучая только небольшую часть параметров.

Главная идея:

```text
Большая pretrained model
        │
        ├── большая часть параметров frozen
        │
        └── небольшая часть параметров trainable
                         │
                         ▼
                      Training
                         │
                         ▼
                   Небольшой adapter
```

Одна базовая модель может использоваться совместно с множеством небольших task-specific adapters.


### Что именно экономит PEFT


PEFT уменьшает:

- число trainable parameters;
- память для gradients;
- память для optimizer states;
- размер task-specific checkpoints;
- стоимость хранения нескольких специализированных версий модели.

Но важно понимать ограничение: frozen base model всё равно участвует в forward pass и обычно должна находиться в GPU memory. PEFT не превращает большую модель в маленькую.


### PEFT как идея и библиотека `peft`


PEFT — это общая концепция, существовавшая до появления библиотеки Hugging Face.

Например, adapter-based parameter-efficient transfer learning для Transformers был опубликован ещё в 2019 году.

Библиотека **`peft`** от Hugging Face предоставляет единый API для большого количества таких методов и интегрируется с `transformers`.

Типичный workflow:

```text
AutoModel...
    │
    ▼
PEFT configuration
    │
    ▼
get_peft_model(...)
    │
    ▼
PeftModel
    │
    ▼
Trainer
    │
    ▼
adapter checkpoint
```


### Основные семейства PEFT


| Семейство | Что обучается | Примеры |
|---|---|---|
| Soft prompting | Виртуальные trainable embeddings | Prompt Tuning, Prefix Tuning, P-Tuning |
| Low-rank adaptation | Небольшие low-rank matrices | LoRA, AdaLoRA |
| Adapter methods | Компактные дополнительные transformations | IA3 и другие adapters |
| Selective tuning | Только выбранные существующие параметры | Trainable tokens, LayerNorm tuning |

LoRA — только один из PEFT-методов, хотя сегодня он является наиболее распространённым.


### Краткая хронология


| Год | Метод | Основная идея |
|---:|---|---|
| 2019 | Adapters | Небольшие trainable modules внутри frozen Transformer |
| 2021 | Prefix Tuning | Trainable continuous prefixes |
| 2021 | Prompt Tuning | Trainable soft prompt embeddings |
| 2021 | P-Tuning | Continuous prompts с prompt encoder |
| 2021 | LoRA | Low-rank decomposition обновления весов |
| 2023 | QLoRA | LoRA поверх 4-bit quantized base model |

В этом notebook используется **Prompt Tuning**, потому что на нём особенно наглядно видно базовый принцип PEFT.


### Hard prompt и soft prompt


Обычная текстовая инструкция — это **hard prompt**:

```text
Classify the sentiment as positive or negative.
```

Она состоит из реальных токенов словаря и выбирается человеком.

**Soft prompt** состоит из trainable vectors в embedding space. Эти vectors не обязаны соответствовать каким-либо словам и изменяются через backpropagation.


### Как работает Prompt Tuning


Пусть обычный tokenized input после embedding layer выглядит так:

```text
x1  x2  x3  ...  xn
```

Prompt Tuning добавляет перед ним обучаемые virtual tokens:

```text
p1  p2  ...  pk  x1  x2  x3  ...  xn
```

где:

```text
p1 ... pk  → trainable
x1 ... xn  → обычные input embeddings
base model → frozen
```

Во время training optimizer изменяет только `p1 ... pk`.


### Full fine-tuning и Prompt Tuning визуально


```text
FULL FINE-TUNING

Task A ──► [ entire model A ]
Task B ──► [ entire model B ]
Task C ──► [ entire model C ]

Для каждой задачи хранится полная модель.


PROMPT TUNING

Prompt A ─┐
Prompt B ─┼──► [ one frozen base model ]
Prompt C ─┘

Для каждой задачи хранится только маленький soft prompt.
```


### Официальная схема Hugging Face


![Model Tuning vs Prompt Tuning](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/prompt-tuning.png)


Схема показывает главное отличие: при обычном model tuning для каждой задачи появляется собственная дообученная модель, тогда как Prompt Tuning позволяет переиспользовать одну frozen base model и хранить маленькие task-specific prompts.


### Сколько параметров обучается


Для простого Prompt Tuning число trainable parameters примерно равно:

```text
num_virtual_tokens × hidden_size
```

В notebook используется:

```text
num_virtual_tokens = 16
hidden_size        = 2048
```

Поэтому ожидаемый порядок величины:

```text
16 × 2048 = 32 768 trainable parameters
```

на фоне примерно **2 миллиардов параметров** Qwen3.5-2B-Base.

Фактическое число notebook затем вычисляет автоматически через `requires_grad`.


### Почему Prompt Tuning масштабируется


Размер soft prompt зависит в основном от:

- количества virtual tokens;
- hidden size модели.

Он почти не зависит от общего количества Transformer layers.

Поэтому при переходе от 2B-модели к более крупной модели количество обучаемых параметров растёт намного медленнее, чем размер base model.


### Почему взят Qwen3.5-2B-Base


`Qwen/Qwen3.5-2B-Base` подходит для учебного PEFT notebook по нескольким причинам:

- это современная base model;
- размер уже достаточно велик, чтобы PEFT имел практический смысл;
- модель всё ещё достаточно компактна для локальных экспериментов;
- text-only backbone доступен через `AutoModelForCausalLM`;
- тот же workflow можно масштабировать на более крупный checkpoint;
- Prompt Tuning почти не зависит от внутренних имён attention modules.

Последний пункт важен: будущий LoRA notebook будет уже сильнее зависеть от внутренней архитектуры модели и выбора `target_modules`.


### Prompt Tuning, LoRA и QLoRA


| Метод | Base model | Что обучается | Quantization |
|---|---|---|---|
| Full fine-tuning | trainable | почти все параметры | не обязательна |
| Prompt Tuning | frozen | virtual prompt embeddings | не обязательна |
| LoRA | frozen | low-rank matrices | не обязательна |
| QLoRA | frozen + quantized | LoRA matrices | обычно 4-bit |

Учебная последовательность:

```text
PEFT / Prompt Tuning
        ↓
       LoRA
        ↓
bitsandbytes quantization
        ↓
      QLoRA
```


### Что проверим на практике


В практической части мы проверим теорию измерениями:

1. сколько параметров содержит Qwen3.5-2B-Base;
2. сколько параметров trainable до PEFT;
3. сколько остаётся trainable после Prompt Tuning;
4. действительно ли base model frozen;
5. качество до обучения;
6. качество после Prompt Tuning;
7. размер adapter checkpoint;
8. загрузку сохранённого adapter поверх исходной модели.


### Ограничения Prompt Tuning


Prompt Tuning обучает очень мало параметров, но это не означает, что он всегда лучший вариант.

- Soft prompts не являются человекочитаемыми.
- Качество может зависеть от initialization.
- Важен выбор числа virtual tokens.
- Base model всё равно требует вычислений.
- Activations всё равно занимают память.
- Для многих современных LLM-задач LoRA обычно показывает более сильный и предсказуемый результат.

Поэтому Prompt Tuning здесь используется прежде всего как максимально наглядное введение в PEFT.


### Источники по теории


- Hugging Face PEFT — Soft prompts: https://huggingface.co/docs/peft/main/en/conceptual_guides/prompting
- Hugging Face PEFT — Methods overview: https://huggingface.co/docs/peft/main/methods/overview
- Parameter-Efficient Transfer Learning for NLP: https://arxiv.org/abs/1902.00751
- The Power of Scale for Parameter-Efficient Prompt Tuning: https://arxiv.org/abs/2104.08691
- LoRA: https://arxiv.org/abs/2106.09685
- QLoRA: https://arxiv.org/abs/2305.14314
- Qwen3.5-2B-Base: https://huggingface.co/Qwen/Qwen3.5-2B-Base


## Практическая часть


## 1. Зависимости


Notebook рассчитан на актуальный стабильный стек Hugging Face на август 2026 года.

PyTorch лучше устанавливать на уровне CUDA-контейнера. Здесь обновляются только библиотеки, необходимые для PEFT.


In [1]:
%pip install -U \
    "transformers==5.15.0" \
    "peft==0.20.0" \
    "datasets==5.0.1" \
    "accelerate==1.14.0"


Note: you may need to restart the kernel to use updated packages.


## 2. Импорты и проверка среды


Проверяем версии библиотек, доступность CUDA и поддержку BF16.


In [2]:
import os
import sys
from dataclasses import dataclass
from importlib.metadata import version as package_version
from pathlib import Path
from typing import Any

import torch
from datasets import DatasetDict, load_dataset
from huggingface_hub import EvalResult, HfApi, ModelCard, ModelCardData, create_repo
from packaging.version import Version
from peft import (
    PeftConfig,
    PeftModel,
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {package_version('transformers')}")
print(f"PEFT:         {package_version('peft')}")
print(f"Datasets:     {package_version('datasets')}")
print(f"Accelerate:   {package_version('accelerate')}")
print(f"Hugging Face Hub: {package_version('huggingface_hub')}")

assert Version(package_version("transformers")) >= Version("5.15.0")
assert Version(package_version("peft")) >= Version("0.20.0")
assert Version(package_version("datasets")) >= Version("5.0.1")
assert Version(package_version("accelerate")) >= Version("1.14.0")

if torch.cuda.is_available():
    print(f"\nGPU:          {torch.cuda.get_device_name(0)}")
    print(f"CUDA:         {torch.version.cuda}")
    print(f"BF16:         {torch.cuda.is_bf16_supported()}")
else:
    print("\nCUDA GPU is not available.")


Python:       3.10.12
PyTorch:      2.13.0+cu130
Transformers: 5.15.0
PEFT:         0.20.0
Datasets:     5.0.1
Accelerate:   1.14.0
Hugging Face Hub: 1.28.0

GPU:          NVIDIA GeForce RTX 5090
CUDA:         13.0
BF16:         True


## 3. Конфигурация и outputs


Train split по умолчанию ограничен 4 000 примерами для быстрого эксперимента.

Validation split **не ограничивается**: `MAX_EVAL_SAMPLES=None`. И baseline, и финальная generation-based оценка выполняются на всём validation split SST-2.

Trainer checkpoints сохраняются отдельно в `outputs/qwen3.5-2b-prompt-tuning/checkpoints/`, а финальный PEFT adapter, tokenizer и Model Card - в корне `outputs/qwen3.5-2b-prompt-tuning/`.


In [3]:
SEED = 42

MODEL_ID = "Qwen/Qwen3.5-2B-Base"
DATASET_ID = "stanfordnlp/sst2"

TEXT_COLUMN = "sentence"
LABEL_COLUMN = "label"
LABEL_NAMES = {0: "negative", 1: "positive"}

PROMPT_INIT_TEXT = "Classify the sentiment of the movie review as positive or negative."
NUM_VIRTUAL_TOKENS = 16

MAX_LENGTH = 128
MAX_TRAIN_SAMPLES = 4_000
MAX_EVAL_SAMPLES = None

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 3e-2
WARMUP_STEPS = 0.05

BASELINE_EVAL_SAMPLES = None
FINAL_EVAL_SAMPLES = None
GENERATION_BATCH_SIZE = 32
MAX_NEW_TOKENS = 4

PUSH_TO_HUB = False
HUB_MODEL_ID = "artyomboyko/qwen3.5-2b-sst2-prompt-tuning"

cwd = Path.cwd().resolve()
if cwd.name == "peft":
    NOTEBOOK_DIR = cwd
elif (cwd / "notebooks" / "finetuning" / "peft").is_dir():
    NOTEBOOK_DIR = (cwd / "notebooks" / "finetuning" / "peft").resolve()
elif Path("/workspace/notebooks/finetuning/peft").is_dir():
    NOTEBOOK_DIR = Path("/workspace/notebooks/finetuning/peft")
else:
    NOTEBOOK_DIR = cwd

OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "qwen3.5-2b-prompt-tuning"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Output directory:     {OUTPUT_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")


Notebook directory: /workspace/notebooks/finetuning/peft
Output directory:     /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-prompt-tuning
Checkpoint directory: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-prompt-tuning/checkpoints


## 4. Загрузка датасета


Используется Stanford SST-2: английские предложения из movie reviews с двумя классами - `negative` и `positive`.

Для первого запуска берётся случайное подмножество train split. Validation используется для оценки.


In [4]:
raw_dataset = load_dataset(DATASET_ID)

dataset = DatasetDict(
    train=raw_dataset["train"],
    validation=raw_dataset["validation"],
)

def limit_split(split, max_samples):
    if max_samples is None:
        return split
    count = min(max_samples, len(split))
    return split.shuffle(seed=SEED).select(range(count))

dataset["train"] = limit_split(dataset["train"], MAX_TRAIN_SAMPLES)
dataset["validation"] = limit_split(dataset["validation"], MAX_EVAL_SAMPLES)

print(dataset)
print(dataset["train"][0])

print(f"Train samples:      {len(dataset['train']):,}")
print(
    f"Validation samples: {len(dataset['validation']):,} "
    f"/ {len(raw_dataset['validation']):,}"
)

if MAX_EVAL_SAMPLES is None:
    assert len(dataset["validation"]) == len(raw_dataset["validation"]), (
        "Validation split was unexpectedly truncated."
    )


DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
})
{'idx': 32326, 'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1}
Train samples:      4,000
Validation samples: 872 / 872


## 5. Tokenizer и базовая модель


Qwen3.5 - нативно мультимодальное семейство, но здесь используется только causal language model backbone через `AutoModelForCausalLM`.

Модель загружается без quantization: 4-bit/8-bit будут рассмотрены отдельно перед QLoRA.


In [5]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
)

print(f"Loaded class: {type(base_model).__name__}")
print(f"Model dtype:  {next(base_model.parameters()).dtype}")
print(f"Vocabulary:   {len(tokenizer):,}")
print(f"EOS token:    {tokenizer.eos_token!r}")
print(f"PAD token:    {tokenizer.pad_token!r}")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded class: Qwen3_5ForCausalLM
Model dtype:  torch.bfloat16
Vocabulary:   248,077
EOS token:    '<|endoftext|>'
PAD token:    '<|endoftext|>'


## 6. Baseline: параметры модели


Считаем общее количество параметров и приблизительный объём весов в текущем dtype.


In [6]:
def parameter_stats(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def parameter_memory_gib(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return total_bytes / (1024 ** 3)

total_params, trainable_params = parameter_stats(base_model)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable share:      {100 * trainable_params / total_params:.6f}%")
print(f"Parameter memory:     {parameter_memory_gib(base_model):.2f} GiB")


Total parameters:     1,881,825,088
Trainable parameters: 1,881,825,088
Trainable share:      100.000000%
Parameter memory:     3.51 GiB


## 7. Формат задачи


Классификация преобразуется в генеративную задачу.

```text
Classify the sentiment of this movie review as positive or negative.
Review: a wonderfully acted and moving story
Sentiment: positive
```

Loss считается только по токенам целевой метки. Инструкция и review маскируются значением `-100`.


In [7]:
VISIBLE_INSTRUCTION = "Classify the sentiment of this movie review as positive or negative."

def build_prompt(text):
    return (
        f"{VISIBLE_INSTRUCTION}\n"
        f"Review: {text.strip()}\n"
        "Sentiment:"
    )

def preprocess_batch(examples):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for text, label_id in zip(examples[TEXT_COLUMN], examples[LABEL_COLUMN]):
        target_text = " " + LABEL_NAMES[int(label_id)]
        target_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]
        target_ids = target_ids + [tokenizer.eos_token_id]

        max_prompt_length = max(1, MAX_LENGTH - len(target_ids))
        prompt_ids = tokenizer(
            build_prompt(text),
            add_special_tokens=False,
            truncation=True,
            max_length=max_prompt_length,
        )["input_ids"]

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        all_input_ids.append(input_ids)
        all_attention_masks.append([1] * len(input_ids))
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }

processed_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=1_000,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing SST-2",
)

print(processed_dataset)


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 872
    })
})


## 8. Data collator


Padding выполняется динамически до максимальной длины текущего batch. Для labels используется `-100`.


In [8]:
@dataclass
class CausalClassificationCollator:
    tokenizer: Any

    def __call__(self, features):
        model_features = [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            return_tensors="pt",
        )

        sequence_length = batch["input_ids"].shape[1]
        padded_labels = []

        for feature in features:
            labels = feature["labels"]
            padding_length = sequence_length - len(labels)
            padded_labels.append(labels + [-100] * padding_length)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

data_collator = CausalClassificationCollator(tokenizer=tokenizer)

test_batch = data_collator(
    [processed_dataset["train"][0], processed_dataset["train"][1]]
)

for key, value in test_batch.items():
    print(f"{key:16s}: {tuple(value.shape)} {value.dtype}")


input_ids       : (2, 37) torch.int64
attention_mask  : (2, 37) torch.int64
labels          : (2, 37) torch.int64


## 9. Baseline inference до PEFT


Перед созданием PEFT-модели измеряем, насколько исходная pre-trained base model уже способна генерировать правильные метки.

Это sanity check, а не основной benchmark.


In [9]:
def normalize_prediction(text):
    text = text.strip().lower()
    if text.startswith("positive"):
        return "positive"
    if text.startswith("negative"):
        return "negative"

    first_line = text.splitlines()[0] if text else ""
    first_word = first_line.split()[0].strip(".,:;!?") if first_line.split() else ""
    return first_word if first_word in {"positive", "negative"} else None

def evaluate_generation_accuracy(model, raw_split, max_samples, batch_size=32):
    model.eval()
    sample_count = (
        len(raw_split)
        if max_samples is None
        else min(max_samples, len(raw_split))
    )
    eval_split = raw_split.select(range(sample_count))

    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    correct = 0
    valid = 0
    predictions = []

    device = next(model.parameters()).device

    for start in range(0, sample_count, batch_size):
        batch_examples = eval_split[start:start + batch_size]
        prompts = [build_prompt(text) for text in batch_examples[TEXT_COLUMN]]

        inputs = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated_ids = outputs[:, inputs["input_ids"].shape[1]:]
        generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for generated_text, label_id in zip(generated_texts, batch_examples[LABEL_COLUMN]):
            prediction = normalize_prediction(generated_text)
            reference = LABEL_NAMES[int(label_id)]

            if prediction is not None:
                valid += 1
                correct += int(prediction == reference)

            predictions.append((generated_text, prediction, reference))

    tokenizer.padding_side = previous_padding_side

    return {
        "accuracy": correct / sample_count,
        "valid_output_rate": valid / sample_count,
        "correct": correct,
        "total": sample_count,
        "examples": predictions,
    }

if torch.cuda.is_available():
    base_model = base_model.to("cuda")

baseline_metrics = evaluate_generation_accuracy(
    base_model,
    dataset["validation"],
    max_samples=BASELINE_EVAL_SAMPLES,
    batch_size=GENERATION_BATCH_SIZE,
)

print(f"Baseline accuracy:       {baseline_metrics['accuracy']:.2%}")
print(f"Valid label output rate: {baseline_metrics['valid_output_rate']:.2%}")

for generated, prediction, reference in baseline_metrics["examples"][:5]:
    print(f"generated={generated!r:20s} parsed={prediction!r:10s} reference={reference}")


Baseline accuracy:       3.10%
Valid label output rate: 3.10%
generated='\n\n<think>\nHmm'   parsed=None       reference=positive
generated='\n\n<think>\nHmm'   parsed=None       reference=negative
generated='\n\n<think>\nHmm'   parsed=None       reference=positive
generated='\n\n<think>\nHmm'   parsed=None       reference=positive
generated='\n\n<think>\nHmm'   parsed=None       reference=negative


## 10. Prompt Tuning configuration


`PromptTuningConfig` добавляет к входу обучаемые virtual tokens.

Веса Qwen3.5 остаются frozen. Такой notebook можно масштабировать на более крупный checkpoint, в основном меняя `MODEL_ID` и batch size.


In [10]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    prompt_tuning_init_text=PROMPT_INIT_TEXT,
    tokenizer_name_or_path=MODEL_ID,
)

model = get_peft_model(base_model, peft_config)
model.config.use_cache = False

total_params, trainable_params = parameter_stats(model)

model.print_trainable_parameters()
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable share:      {100 * trainable_params / total_params:.8f}%")


trainable params: 32,768 || all params: 1,881,857,856 || trainable%: 0.0017

Total parameters:     1,881,857,856
Trainable parameters: 32,768
Trainable share:      0.00174126%


## 11. Проверка frozen base model


После `get_peft_model()` trainable должны остаться только параметры Prompt Tuning.


In [11]:
trainable_names = [
    name for name, parameter in model.named_parameters() if parameter.requires_grad
]

print(f"Number of trainable tensors: {len(trainable_names)}")
for name in trainable_names:
    print(name)

assert trainable_params < total_params * 0.001, "Too many parameters are trainable for Prompt Tuning."


Number of trainable tensors: 1
prompt_encoder.default.embedding.weight


## 12. TrainingArguments


Prompt Tuning обычно использует более высокий learning rate, чем full fine-tuning или LoRA.

На совместимых NVIDIA GPU используется BF16 и fused AdamW.


In [12]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.0,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    bf16=use_bf16,
    fp16=use_fp16,
    tf32=True if torch.cuda.is_available() else None,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",

    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=torch.cuda.is_available(),

    report_to="none",

    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
)

print("BF16:", use_bf16)
print("FP16:", use_fp16)
print("Output:", OUTPUT_DIR)


BF16: True
FP16: False
Output: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-prompt-tuning


## 13. Trainer


Используется стандартный `transformers.Trainer`. Optimizer получает только параметры с `requires_grad=True`.


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)


## 14. Обучение


Для продолжения прерванного запуска можно использовать:

```python
trainer.train(resume_from_checkpoint=True)
```


In [14]:
train_result = trainer.train()

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Epoch,Training Loss,Validation Loss
1,0.098130,0.084762
2,0.068929,0.089276
3,0.056976,0.083539


***** train metrics *****
  epoch                    =        3.0
  total_flos               =  4969010GF
  train_loss               =     0.1076
  train_runtime            = 0:06:44.88
  train_samples_per_second =     29.638
  train_steps_per_second   =      1.852


## 15. Evaluation loss


Выполняется обычная оценка causal-LM loss на validation split.


In [15]:
eval_metrics = trainer.evaluate()

trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

eval_metrics


Training Loss,Validation Loss,Epoch
0.056976,0.083539,3


***** eval metrics *****
  eval_loss = 0.0835


{'eval_loss': 0.08353926986455917}

## 16. Accuracy после Prompt Tuning


Повторяется тот же generation-based sanity check, который использовался для исходной base model.


In [16]:
model = trainer.model
model.config.use_cache = True

final_metrics = evaluate_generation_accuracy(
    model,
    dataset["validation"],
    max_samples=FINAL_EVAL_SAMPLES,
    batch_size=GENERATION_BATCH_SIZE,
)

print(f"Evaluation samples:      {final_metrics['total']:,}")
print(f"Baseline accuracy:       {baseline_metrics['accuracy']:.2%}")
print(f"Prompt-tuned accuracy:   {final_metrics['accuracy']:.2%}")
print(f"Valid label output rate: {final_metrics['valid_output_rate']:.2%}")

for generated, prediction, reference in final_metrics["examples"][:10]:
    print(f"generated={generated!r:20s} parsed={prediction!r:10s} reference={reference}")


/usr/local/lib/python3.10/dist-packages/peft/peft_model.py:1657: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


Evaluation samples:      872
Baseline accuracy:       3.10%
Prompt-tuned accuracy:   94.38%
Valid label output rate: 100.00%
generated=' positive'          parsed='positive' reference=positive
generated=' negative'          parsed='negative' reference=negative
generated=' positive'          parsed='positive' reference=positive
generated=' positive'          parsed='positive' reference=positive
generated=' negative'          parsed='negative' reference=negative
generated=' positive'          parsed='positive' reference=positive
generated=' negative'          parsed='negative' reference=negative
generated=' negative'          parsed='negative' reference=negative
generated=' positive'          parsed='positive' reference=positive
generated=' negative'          parsed='negative' reference=negative


## 17. Сохранение PEFT adapter


В `outputs/qwen3.5-2b-prompt-tuning/` сохраняется не полная Qwen3.5-2B, а только Prompt Tuning adapter и tokenizer.

Базовые веса остаются в общем Hugging Face cache.


In [17]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved PEFT adapter to: {OUTPUT_DIR}")


Saved PEFT adapter to: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-prompt-tuning


## 18. Размер адаптера


Сравниваем размер task-specific adapter с приблизительным объёмом параметров base model.


In [18]:
adapter_files = [
    OUTPUT_DIR / "adapter_model.safetensors",
    OUTPUT_DIR / "adapter_config.json",
]

adapter_size_bytes = sum(
    path.stat().st_size
    for path in adapter_files
    if path.exists()
)

print(f"PEFT adapter size:      {adapter_size_bytes / 1024**2:.3f} MiB")
print(f"Base parameter memory:  {parameter_memory_gib(base_model):.2f} GiB")

print("\nFinal export files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(f"{file.name:32s} {file.stat().st_size / 1024:.1f} KiB")


PEFT adapter size:      0.126 MiB
Base parameter memory:  3.51 GiB

Final export files:
README.md                        5.1 KiB
adapter_config.json              0.6 KiB
adapter_model.safetensors        128.1 KiB
chat_template.jinja              7.6 KiB
tokenizer.json                   19521.1 KiB
tokenizer_config.json            1.1 KiB


## 19. Загрузка сохранённого адаптера


Для inference нужны base model и PEFT adapter.

Ячейка по умолчанию не загружает вторую копию Qwen3.5, чтобы не занимать GPU memory.


In [19]:
RELOAD_ADAPTER = False

if RELOAD_ADAPTER:
    adapter_config = PeftConfig.from_pretrained(OUTPUT_DIR)

    reloaded_base_model = AutoModelForCausalLM.from_pretrained(
        adapter_config.base_model_name_or_path,
        dtype=model_dtype,
    )

    reloaded_model = PeftModel.from_pretrained(
        reloaded_base_model,
        OUTPUT_DIR,
    )

    if torch.cuda.is_available():
        reloaded_model = reloaded_model.to("cuda")

    reloaded_model.eval()

    print(type(reloaded_model).__name__)
    print("Adapter reloaded successfully.")
else:
    print("Set RELOAD_ADAPTER = True to test a clean adapter reload.")


Set RELOAD_ADAPTER = True to test a clean adapter reload.


## 20. Создание Model Card


Model Card создаётся автоматически из фактических параметров текущего запуска.

В `README.md` записываются:

- base model и PEFT method;
- dataset и evaluation split;
- число trainable parameters;
- training configuration;
- baseline и prompt-tuned accuracy;
- valid label output rate;
- точное число evaluation samples;
- пример загрузки adapter;
- limitations и intended use.

В metadata также создаётся `model-index` с accuracy на SST-2 validation, чтобы результат корректно отображался на Hugging Face Hub.


In [20]:
MODEL_CARD_PATH = OUTPUT_DIR / "README.md"

evaluation_is_full_validation = (
    final_metrics["total"] == len(raw_dataset["validation"])
)

evaluation_scope = (
    "the full SST-2 validation split"
    if evaluation_is_full_validation
    else f"a subset of {final_metrics['total']} SST-2 validation examples"
)

model_display_name = HUB_MODEL_ID.split("/")[-1]

eval_result = EvalResult(
    task_type="text-classification",
    task_name="Sentiment Classification via Text Generation",
    dataset_type=DATASET_ID,
    dataset_name="Stanford SST-2",
    dataset_split="validation",
    metric_type="accuracy",
    metric_name="Generation-based Accuracy",
    metric_value=round(final_metrics["accuracy"], 6),
)

card_data = ModelCardData(
    base_model=MODEL_ID,
    datasets=[DATASET_ID],
    eval_results=[eval_result],
    language="en",
    library_name="peft",
    license="apache-2.0",
    metrics=["accuracy"],
    model_name=model_display_name,
    pipeline_tag="text-generation",
    tags=[
        "peft",
        "prompt-tuning",
        "qwen3.5",
        "sentiment-analysis",
        "text-classification",
        "causal-lm",
    ],
)

gpu_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

trainable_share = 100.0 * trainable_params / total_params

card_text = f"""---
{card_data.to_yaml()}
---

# Qwen3.5-2B SST-2 Prompt Tuning

Parameter-efficient adaptation of
[`{MODEL_ID}`](https://huggingface.co/{MODEL_ID})
for binary sentiment classification on
[`{DATASET_ID}`](https://huggingface.co/datasets/{DATASET_ID})
using **Prompt Tuning**.

> This repository contains a PEFT Prompt Tuning adapter, not the full
> Qwen3.5-2B-Base model. The base model is required for inference.

## Model Details

| Property | Value |
|---|---|
| Base model | `{MODEL_ID}` |
| PEFT method | Prompt Tuning |
| Task | Binary sentiment classification via text generation |
| Dataset | `{DATASET_ID}` |
| Labels | `negative`, `positive` |
| Virtual tokens | {NUM_VIRTUAL_TOKENS} |
| Prompt initialization | `{PROMPT_INIT_TEXT}` |
| Trainable parameters | {trainable_params:,} |
| Total parameters | {total_params:,} |
| Trainable share | {trainable_share:.6f}% |

The Qwen3.5 base-model weights remain frozen. Only the learned virtual
prompt embeddings are updated during training.

## Results

Evaluation was performed on **{evaluation_scope}**.

| Metric | Base model | Prompt-tuned adapter |
|---|---:|---:|
| Generation-based accuracy | {baseline_metrics["accuracy"]:.2%} | **{final_metrics["accuracy"]:.2%}** |
| Valid label output rate | {baseline_metrics["valid_output_rate"]:.2%} | **{final_metrics["valid_output_rate"]:.2%}** |
| Evaluation examples | {baseline_metrics["total"]:,} | {final_metrics["total"]:,} |

A prediction is counted as correct only when the generated output can be
parsed as exactly one of the task labels (`positive` or `negative`) and
matches the SST-2 reference label.

The base-model result should therefore not be interpreted as a general
measure of Qwen3.5 capability. It measures zero-shot performance under
this specific generation format.

## Training Configuration

| Parameter | Value |
|---|---:|
| Training examples | {len(dataset["train"]):,} |
| Validation examples | {len(dataset["validation"]):,} |
| Epochs | {NUM_TRAIN_EPOCHS} |
| Learning rate | {LEARNING_RATE} |
| Warmup fraction | {WARMUP_STEPS} |
| Train batch size / device | {TRAIN_BATCH_SIZE} |
| Eval batch size / device | {EVAL_BATCH_SIZE} |
| Gradient accumulation | {GRADIENT_ACCUMULATION_STEPS} |
| Maximum sequence length | {MAX_LENGTH} |
| Precision | {"BF16" if use_bf16 else ("FP16" if use_fp16 else "FP32")} |
| Hardware | {gpu_name} |

## Prompt Format

```text
{VISIBLE_INSTRUCTION}
Review: <SST-2 sentence>
Sentiment:
```

The expected generated continuation is either:

```text
 positive
```

or:

```text
 negative
```

## Usage

```python
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "{MODEL_ID}"
ADAPTER_ID = "{HUB_MODEL_ID}"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=dtype,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
)

if torch.cuda.is_available():
    model = model.to("cuda")

model.eval()

review = "a wonderfully acted and moving story"
prompt = (
    "{VISIBLE_INSTRUCTION}\\n"
    f"Review: {{review}}\\n"
    "Sentiment:"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens={MAX_NEW_TOKENS},
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = outputs[:, inputs["input_ids"].shape[1]:]
prediction = tokenizer.batch_decode(
    generated,
    skip_special_tokens=True,
)[0].strip()

print(prediction)
```

## Intended Use

This adapter is intended as an educational and reproducible example of
Parameter-Efficient Fine-Tuning with Prompt Tuning on a modern
billion-parameter language model.

It can also be used for experiments with SST-2 sentiment classification
under the prompt format documented above.

## Limitations

- The adapter is specialized for English SST-2 sentiment classification.
- It was trained on {len(dataset["train"]):,} training examples, not the complete SST-2 train split.
- Classification is implemented through causal text generation rather than a dedicated classification head.
- The adapter expects the prompt format shown above.
- Soft prompts are not human-readable instructions.
- Results are specific to this dataset, prompt format, hyperparameters, and random seed.
- The adapter requires `{MODEL_ID}` and is not a standalone model.
- Further evaluation is required before using the adapter in production applications.

## Reproducibility

| Software | Version |
|---|---|
| PyTorch | {torch.__version__} |
| Transformers | {package_version("transformers")} |
| PEFT | {package_version("peft")} |
| Datasets | {package_version("datasets")} |
| Accelerate | {package_version("accelerate")} |
| huggingface_hub | {package_version("huggingface_hub")} |

Random seed: `{SEED}`.

## References

- [Qwen3.5-2B-Base](https://huggingface.co/{MODEL_ID})
- [Stanford SST-2](https://huggingface.co/datasets/{DATASET_ID})
- [Hugging Face PEFT](https://huggingface.co/docs/peft/)
- Lester et al., *The Power of Scale for Parameter-Efficient Prompt Tuning* (2021)

## License

The base model is distributed under the Apache-2.0 license. This adapter
uses the same license metadata.
"""

model_card = ModelCard(card_text)
model_card.save(MODEL_CARD_PATH)

try:
    model_card.validate()
    print("Model Card metadata validation: OK")
except Exception as error:
    print(f"Model Card validation warning: {error}")

print(f"Model Card saved to: {MODEL_CARD_PATH}")
print(
    f"Evaluation scope: "
    f"{'full validation split' if evaluation_is_full_validation else 'subset'} "
    f"({final_metrics['total']:,} examples)"
)


Model Card metadata validation: OK
Model Card saved to: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-prompt-tuning/README.md
Evaluation scope: full validation split (872 examples)


## 21. Публикация в Hugging Face Hub


Push в Hub отключён по умолчанию.

Перед публикацией выполните `hf auth login` и установите `PUSH_TO_HUB=True`.

Публикуется содержимое финального `OUTPUT_DIR`: PEFT adapter, tokenizer и созданный `README.md`. Trainer checkpoints из `checkpoints/` на Hub не отправляются.


In [23]:
if PUSH_TO_HUB:
    create_repo(
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        exist_ok=True,
    )

    api = HfApi()

    commit_info = api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        ignore_patterns=[
            "checkpoints/**",
            "checkpoint-*/**",
            "runs/**",
            "*.pt",
            "*.pth",
        ],
        commit_message="Upload Prompt Tuning adapter and model card",
    )

    print(f"Published: https://huggingface.co/{HUB_MODEL_ID}")
    print(f"Commit:    {commit_info.commit_url}")
else:
    print("PUSH_TO_HUB=False - nothing was uploaded.")


Published: https://huggingface.co/artyomboyko/qwen3.5-2b-sst2-prompt-tuning
Commit:    https://huggingface.co/artyomboyko/qwen3.5-2b-sst2-prompt-tuning/commit/0555a19b94a19cb614ff72536a8c0b3c9ca47f54


## 22. Что показал этот notebook


После выполнения должны быть видны основные свойства PEFT:

1. Base model содержит около 2B параметров.
2. При Prompt Tuning базовые веса модели заморожены.
3. Обучается только очень малая доля параметров - virtual prompt embeddings.
4. Generation-based evaluation выполняется на полном SST-2 validation split.
5. Сохраняется небольшой adapter, который загружается поверх исходной Qwen3.5.
6. Для adapter автоматически создаётся полноценный Hugging Face Model Card с фактическими метриками запуска.


## Источники


- Qwen3.5-2B-Base: https://huggingface.co/Qwen/Qwen3.5-2B-Base
- Transformers Qwen3.5: https://huggingface.co/docs/transformers/model_doc/qwen3_5
- PEFT Prompt Tuning: https://huggingface.co/docs/peft/main/package_reference/prompt_tuning
- PEFT causal LM Prompt Tuning guide: https://huggingface.co/docs/peft/main/task_guides/clm-prompt-tuning
- Stanford SST-2: https://huggingface.co/datasets/stanfordnlp/sst2
